Example notebook for the Young Stellar Objects metric. 
Please contact Loredo Prisidano for more information about the details of the metric. 

The basic premise is to count how many young stellar objects above a SNR cut in g,r, and i bands are available in the coadded images. Dust extinction and distance modulus are taken into account, when searching for stars above a given absolute magnitude. 

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

%matplotlib inline

import rubin_sim.maf as maf
from rubin_sim.data import get_baseline

In [ ]:
baseline_file = get_baseline()
run_name = os.path.basename(baseline_file).replace(".db", "")
outDir = "temp"
resultsDb = maf.db.ResultsDb(out_dir=outDir)

In [ ]:
# Set up YSO metric - which counts the number of expected young stellar objects detectable with Rubin

nside = 64
bundleList = []
sql = ""
# Let's plug in the magnitudes for one type
metric = maf.maf_contrib.NYoungStarsMetric()

print("Metric is looking for YSO with the following absolute magnitudes", metric.mags)

slicer = maf.slicers.HealpixSlicer(nside=nside, use_cache=False)

summaryStats = [maf.metrics.SumMetric(), maf.metrics.MaxMetric()]
plot_dict = {
    "log_scale": True,
    "figsize": (8, 6),
    "color_min": 1,
    "colorMax": 1e5,
    "extend": "both",
    "cbar_format": "%.0e",
}
bundleList.append(
    maf.metricBundles.MetricBundle(
        metric, slicer, sql, plot_dict=plot_dict, summary_metrics=summaryStats, run_name=run_name
    )
)

In [ ]:
bd = maf.metricBundles.make_bundles_dict_from_list(bundleList)
bg = maf.metricBundles.MetricBundleGroup(
    bd, baseline_file, out_dir=outDir, results_db=resultsDb, verbose=True
)
bg.run_all()

In [ ]:
bg.plot_all()

In [ ]:
pd.DataFrame([bd[k].summary_values for k in bd], index=list(bd.keys()))

In [ ]:
b = bundleList[0]
print(
    np.sum(b.metric_values.compressed()),
    np.median(b.metric_values.compressed()),
    np.mean(b.metric_values.compressed()),
    np.max(b.metric_values.compressed()),
    np.min(b.metric_values.compressed()),
    np.std(b.metric_values.compressed()),
)